In [1]:
from pathlib import Path

import numpy as np

from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe


DATA_FILE = Path("data/raw/zeros1")

assert DATA_FILE.exists(), f"Missing data file: {DATA_FILE}"

print("data:", DATA_FILE)
print("numpy:", np.__version__)
print("OK")

data: data/raw/zeros1
numpy: 2.1.3
OK


In [2]:
gamma = np.loadtxt(DATA_FILE, dtype=np.float64)

print("N:", len(gamma))
print("first:", gamma[:5])
print("last:", gamma[-5:])

N: 100000
first: [14.13472514 21.02203964 25.01085758 30.42487613 32.93506159]
last: [74918.37058023 74918.69143345 74919.07516112 74920.25979326
 74920.82749899]


In [3]:
assert gamma.ndim == 1
assert gamma.dtype == np.float64
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

print("shape:", gamma.shape)
print("dtype:", gamma.dtype)
print("finite:", np.all(np.isfinite(gamma)))
print("strictly increasing:", np.all(np.diff(gamma) > 0))
print("OK")

shape: (100000,)
dtype: float64
finite: True
strictly increasing: True
OK


In [4]:
delta = spacings(gamma)

assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("zeros:", len(gamma))
print("spacings:", len(delta))
print("first spacings:", delta[:10])

zeros: 100000
spacings: 99999
first spacings: [6.8873145  3.98881794 5.41401855 2.51018546 4.65111657 3.33254085
 2.40835427 4.6780776  1.7686816  3.196489  ]


In [5]:
print("gamma:")
print(describe(gamma))

print("\ndelta:")
print(describe(delta))

gamma:
{'n': 100000, 'min': 14.134725142, 'max': 74920.827498994, 'mean': 39693.702451121906, 'std': 21075.935845302272}

delta:
{'n': 99999, 'min': 0.014701476000482216, 'max': 6.887314496999998, 'mean': 0.7490744184827048, 'std': 0.32154023373868434}


In [6]:
gamma_range = gamma[-1] - gamma[0]
mean_spacing = np.mean(delta)
range_per_spacing = gamma_range / len(delta)

print("range:", gamma_range)
print("mean spacing:", mean_spacing)
print("range / number of spacings:", range_per_spacing)

assert np.isclose(mean_spacing, range_per_spacing)

range: 74906.692773852
mean spacing: 0.7490744184827048
range / number of spacings: 0.7490744184827048


In [7]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
percentiles = np.percentile(delta, percentile_levels)

for p, value in zip(percentile_levels, percentiles):
    print(f"{p:>3}% : {value:.12f}")

  0% : 0.014701476000
  1% : 0.171940818789
  5% : 0.290969494202
 25% : 0.522532652500
 50% : 0.714470404000
 75% : 0.935617602001
 95% : 1.318236503501
 99% : 1.656737500120
100% : 6.887314497000


In [8]:
BLOCK_SIZE = 1000

num_blocks = len(delta) // BLOCK_SIZE

block_means = np.array([
    np.mean(delta[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE])
    for i in range(num_blocks)
])

remainder_delta = delta[num_blocks * BLOCK_SIZE:]

print("block size:", BLOCK_SIZE)
print("full blocks:", num_blocks)
print("remainder:", len(remainder_delta))

print("\nfirst block means:")
print(block_means[:10])

print("\nlast block means:")
print(block_means[-10:])

block size: 1000
full blocks: 99
remainder: 999

first block means:
[1.4062818  1.09615452 1.01749154 0.97368259 0.94116323 0.91776294
 0.8992915  0.8830238  0.87108584 0.85858228]

last block means:
[0.67626702 0.67593563 0.6755959  0.67383326 0.67353449 0.67329096
 0.67236037 0.67175373 0.67056177 0.6707861 ]


In [9]:
print("local mean spacing:")
print("  min :", block_means.min())
print("  max :", block_means.max())
print("  mean:", block_means.mean())
print("  std :", block_means.std())

print("\nratio max/min:", block_means.max() / block_means.min())

local mean spacing:
  min : 0.6705617749410012
  max : 1.406281801182
  mean: 0.7498774776000706
  std : 0.10289785701303332

ratio max/min: 2.0971696475029917


In [10]:
local_residuals = np.empty_like(delta)

for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE

    local_residuals[start:end] = delta[start:end] - block_means[i]

if len(remainder_delta):
    start = num_blocks * BLOCK_SIZE
    local_residuals[start:] = remainder_delta - np.mean(remainder_delta)

assert local_residuals.shape == delta.shape

print("local-mean residual:")
print("  std :", np.std(local_residuals))
print("  min :", np.min(local_residuals))
print("  max :", np.max(local_residuals))

local-mean residual:
  std : 0.3046998825116909
  min : -1.2447810131819879
  max : 5.481032695817999


In [11]:
predicted_global = gamma[0] + np.arange(len(gamma)) * mean_spacing
residual_global = gamma - predicted_global

print("global constant-spacing baseline:")
print("  residual std:", np.std(residual_global))
print("  residual min:", np.min(residual_global))
print("  residual max:", np.max(residual_global))

global constant-spacing baseline:
  residual std: 958.6860077538355
  residual min: -0.25418903602985665
  residual max: 3277.444140274507


In [12]:
predicted_delta = np.empty_like(delta)

for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE

    predicted_delta[start:end] = block_means[i]

if len(remainder_delta):
    predicted_delta[num_blocks * BLOCK_SIZE:] = np.mean(remainder_delta)

residual = delta - predicted_delta

assert predicted_delta.shape == delta.shape
assert residual.shape == delta.shape

print("local block baseline:")
print("  residual std:", np.std(residual))
print("  residual min:", np.min(residual))
print("  residual max:", np.max(residual))

local block baseline:
  residual std: 0.3046998825116909
  residual min: -1.2447810131819879
  residual max: 5.481032695817999


In [13]:
epsilons = [
    1e-1,
    5e-2,
    1e-2,
    5e-3,
    1e-3,
]

for eps in epsilons:
    q = np.rint(delta / eps).astype(np.int64)
    reconstructed = q * eps

    error = delta - reconstructed
    max_error = np.max(np.abs(error))

    q_min = q.min()
    q_max = q.max()

    levels = q_max - q_min + 1
    bits = np.ceil(np.log2(levels))

    print(
        f"eps={eps:g}  "
        f"bits/value≈{bits:.0f}  "
        f"max_error={max_error:.6g}"
    )

eps=0.1  bits/value≈7  max_error=0.05
eps=0.05  bits/value≈8  max_error=0.0249988
eps=0.01  bits/value≈10  max_error=0.00499998
eps=0.005  bits/value≈11  max_error=0.00249999
eps=0.001  bits/value≈13  max_error=0.000499994


In [14]:
print("=== 02_describe summary ===")

print("zeros          :", len(gamma))
print("spacings       :", len(delta))
print("gamma range    :", gamma[-1] - gamma[0])
print("mean spacing   :", np.mean(delta))
print("spacing std    :", np.std(delta))
print("spacing min    :", np.min(delta))
print("spacing max    :", np.max(delta))
print("block size     :", BLOCK_SIZE)
print("local blocks   :", len(block_means))

print("\nALL BASIC INVARIANTS PASSED")

=== 02_describe summary ===
zeros          : 100000
spacings       : 99999
gamma range    : 74906.692773852
mean spacing   : 0.7490744184827048
spacing std    : 0.32154023373868434
spacing min    : 0.014701476000482216
spacing max    : 6.887314496999998
block size     : 1000
local blocks   : 99

ALL BASIC INVARIANTS PASSED


In [15]:
from pathlib import Path

import numpy as np

from nicht_riemann_data.transforms import spacings


DATA_FILE = Path("data/raw/zeros1")

assert DATA_FILE.exists(), f"Missing data file: {DATA_FILE}"

print("data:", DATA_FILE)
print("numpy:", np.__version__)
print("OK")

data: data/raw/zeros1
numpy: 2.1.3
OK


In [16]:
gamma = np.loadtxt(DATA_FILE, dtype=np.float64)

assert gamma.ndim == 1
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)

print("zeros:", len(gamma))
print("first:", gamma[:5])
print("last:", gamma[-5:])

zeros: 100000
first: [14.13472514 21.02203964 25.01085758 30.42487613 32.93506159]
last: [74918.37058023 74918.69143345 74919.07516112 74920.25979326
 74920.82749899]


In [17]:
delta = spacings(gamma)

assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)

print("spacings:", len(delta))

spacings: 99999


In [18]:
BLOCK_SIZE = 1000

num_blocks = len(delta) // BLOCK_SIZE

local_mean = np.empty_like(delta)

for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE

    local_mean[start:end] = np.mean(delta[start:end])

remainder_start = num_blocks * BLOCK_SIZE

if remainder_start < len(delta):
    local_mean[remainder_start:] = np.mean(delta[remainder_start:])

assert local_mean.shape == delta.shape
assert np.all(local_mean > 0)

print("blocks:", num_blocks)
print("local scale min:", local_mean.min())
print("local scale max:", local_mean.max())

blocks: 99
local scale min: 0.669491983428425
local scale max: 1.406281801182


In [19]:
unfolded = delta / local_mean

assert unfolded.shape == delta.shape
assert np.all(np.isfinite(unfolded))
assert np.all(unfolded > 0)

print("unfolded spacings:", len(unfolded))
print("mean:", np.mean(unfolded))
print("std :", np.std(unfolded))
print("min :", np.min(unfolded))
print("max :", np.max(unfolded))

unfolded spacings: 99999
mean: 0.9999999999999997
std : 0.40186713998930124
min : 0.02186547079062141
max : 4.897535110822818


In [20]:
print("RAW")
print("  mean:", np.mean(delta))
print("  std :", np.std(delta))
print("  min :", np.min(delta))
print("  max :", np.max(delta))

print("\nUNFOLDED")
print("  mean:", np.mean(unfolded))
print("  std :", np.std(unfolded))
print("  min :", np.min(unfolded))
print("  max :", np.max(unfolded))

RAW
  mean: 0.7490744184827048
  std : 0.32154023373868434
  min : 0.014701476000482216
  max : 6.887314496999998

UNFOLDED
  mean: 0.9999999999999997
  std : 0.40186713998930124
  min : 0.02186547079062141
  max : 4.897535110822818


In [21]:
unfolded_block_means = np.array([
    np.mean(unfolded[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE])
    for i in range(num_blocks)
])

print("first:", unfolded_block_means[:10])
print("last :", unfolded_block_means[-10:])

print("\nmean:", unfolded_block_means.mean())
print("std :", unfolded_block_means.std())

first: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
last : [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

mean: 1.0
std : 6.409875621278546e-17


In [22]:
assert np.isclose(
    np.mean(unfolded[:num_blocks * BLOCK_SIZE].reshape(num_blocks, BLOCK_SIZE), axis=1).mean(),
    1.0,
    atol=1e-12,
)

print("local normalization invariant: OK")

local normalization invariant: OK


In [23]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
percentiles = np.percentile(unfolded, percentile_levels)

for p, value in zip(percentile_levels, percentiles):
    print(f"{p:>3}% : {value:.12f}")

  0% : 0.021865470791
  1% : 0.234538581554
  5% : 0.398679540794
 25% : 0.710630119302
 50% : 0.965046539834
 75% : 1.252004821079
 95% : 1.720129557467
 99% : 2.072834102416
100% : 4.897535110823


In [24]:
print("=== 03_unfold summary ===")
print("raw spacings    :", len(delta))
print("unfolded        :", len(unfolded))
print("block size      :", BLOCK_SIZE)
print("unfolded mean   :", np.mean(unfolded))
print("unfolded std    :", np.std(unfolded))

print("\nALL BASIC INVARIANTS PASSED")

=== 03_unfold summary ===
raw spacings    : 99999
unfolded        : 99999
block size      : 1000
unfolded mean   : 0.9999999999999997
unfolded std    : 0.40186713998930124

ALL BASIC INVARIANTS PASSED
